Installing all the necessary dependencies


In [ ]:
!pip install langchain  langchain-core langchain-community transformers
!pip install docling
!pip install -qU langchain-docling
!pip install -qU langchain-text-splitters
!pip install chromadb
!pip install rank-bm25
!pip install torch

#DATA INGESETION PIPELINE



LOADING THE DATASET


In [ ]:
#LOADING DATASET EITHER WAY
# import kagglehub

# # Download latest version
# path = kagglehub.competition_download('ra-gnarok')

# print("Path to competition files:", path)

In [ ]:
from langchain_docling.loader import DoclingLoader,ExportType
FILE_PATH = '/content/EU AI Act.pdf'
loader = DoclingLoader(FILE_PATH , export_type=ExportType.MARKDOWN)

In [ ]:
docs=loader.load()

In [ ]:
md_file = docs[0].page_content
print(md_file)

#PREPROCESSING PIPELINE


In [ ]:
import re

def remove_escaped_underscores(md: str) -> str:
  return re.sub(r'(\\_){3,}', '', md)

def remove_extra_blank_lines(md: str) -> str:
    """
    Replaces multiple consecutive blank lines with a single blank line.
    """
    return re.sub(r'\n\s*\n+', '\n\n', md).strip()

def remove_fake_tables(md: str) -> str:
    lines = md.splitlines()
    cleaned = []

    for line in lines:
        # Skip Markdown table separator rows
        if re.match(r'^\|\s*-+\s*\|\s*-+\s*\|$', line):
            continue

        # Process two-column table rows
        m = re.match(r'^\|\s*(.*?)\s*\|\s*(.*?)\s*\|$', line)
        if m:
            left, right = m.groups()

            # If left side is just '-' or empty, keep only right
            if left.strip() in {"", "-"}:
                cleaned.append(right.strip())
            else:
                # Keep both columns as plain text
                cleaned.append(left.strip())
                if right.strip() and right.strip() != left.strip():
                    cleaned.append(right.strip())
        else:
            cleaned.append(line)

    return "\n".join(cleaned)

def remove_consecutive_duplicates(md: str) -> str:
    lines = md.splitlines()

    cleaned = []
    prev = None

    for line in lines:
        if line.strip() == prev:
            continue

        cleaned.append(line)
        prev = line.strip()

    return "\n".join(cleaned)

In [ ]:
cleaned_md = remove_escaped_underscores(md_file)
cleaned_md = remove_extra_blank_lines(cleaned_md)
cleaned_md = remove_fake_tables(cleaned_md)
cleaned_md = remove_consecutive_duplicates(cleaned_md)

#CHUNKING STRATEGY

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "chapter"),
    ("##", "article"),
    ("###", "section"),
    ("####", "subsection"),
    ("#####", "subsubsection"),
]



In [ ]:
markdown_splitter =  MarkdownHeaderTextSplitter(headers_to_split_on)
chunk_docs = markdown_splitter.split_text(cleaned_md)
print(type(chunk_docs))
print(len(chunk_docs))

APPLYING SECOND CHUNKING FOR BETTER CHUNKS


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

rc_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap=200
)
new_chunk_docs = rc_splitter.split_documents(chunk_docs)
print(len(chunk_docs))
print(len(new_chunk_docs))

#VEC DB & EMBEDDING MODEL

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

embedding_function = SentenceTransformerEmbeddingFunction(model_name="BAAI/bge-m3")
client = chromadb.Client()
collection = client.create_collection(
    name ="EU_AI_Act" ,
    embedding_function=embedding_function,
    configuration = {
       "hnsw": {
            "space": "cosine",
            "ef_construction": 200,
            "max_neighbors":32
        }
    })

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-m3")

In [ ]:
# collection.add(
#  ids = [str(i) for i in range(1,len(new_chunk_docs)+1)],
#  documents=[i.page_content for i in new_chunk_docs],
#  metadatas= [i.metadata for i in new_chunk_docs]
# )
# taking too much time to insert the record

In [ ]:
#embedding generation  and adding to collection

counter = 1
for i in new_chunk_docs:
  embedding = model.encode([i.page_content])
  collection.add(
      ids = str(counter),
      documents = i.page_content,
      metadatas = i.metadata,
      embeddings = embedding
  )
  print("record inserted successfully" , counter)
  counter = counter +1


ADDING BM25


In [ ]:
from langchain_community.retrievers import BM25Retriever
bm25_retriver = BM25Retriever.from_documents(
    documents=new_chunk_docs
    )
bm25_retriver.k=100

ADDING SPLADE


In [ ]:
from sentence_transformers import SparseEncoder
sp_model = SparseEncoder("sparse-encoder-testing/splade-bert-tiny-nq-onnx")

In [ ]:
documents_db=[i.page_content for i in new_chunk_docs]

In [ ]:
document_embeddings = sp_model.encode(
    inputs=documents_db
)

QUERY AND RETRIVE

In [ ]:
import torch
query = "If a company develops an AI model trained with 10^26 FLOPs and integrates it into a customer-facing chatbot sold in the EU, what obligations does the company face under the AI Act?"


#dense retrival serach

dense_res  = collection.query(
    query_texts=query,
    n_results=100
)

#bm25 results
bm25_res = bm25_retriver.invoke(query)

#splade results
sp_empedding = sp_model.encode(query)
similarity_scores = sp_model.similarity(
      document_embeddings,
      sp_empedding

)

# Remove unnecessary dimension
scores = similarity_scores.squeeze()

# Get Top-K scores and their indices
top_k = 100

top_scores, top_indices = torch.topk(
    scores,
    k=top_k
)

# Get the corresponding documents
res_list = [
    documents_db[i]
    for i in top_indices.tolist()
]


In [ ]:
print(type(dense_res))
print(type(bm25_res))
print(type(res_list))

COMBINING RESULTS
(RRF)


In [ ]:
from collections import defaultdict


def reciprocal_rank_fusion(
    dense_results,
    bm25_results,
    splade_results,
    k=60
):
    rrf_scores = defaultdict(float)

    result_lists = [
        dense_results,
        bm25_results,
        splade_results
    ]

    for results in result_lists:

        for rank, document in enumerate(results, start=1):

            rrf_scores[document] += 1 / (k + rank)

    ranked_results = sorted(
        rrf_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked_results

In [ ]:
dense_result = dense_res["documents"]
bm25_res =[i.page_content for i in bm25_res]
#CALLING
rrf_results = reciprocal_rank_fusion(
    dense_result[0],
    bm25_res,
    res_list
)

In [ ]:
top_80_documents = [
    document
    for document, score in rrf_results[:80]
]

RERANKING


In [ ]:
from sentence_transformers import CrossEncoder

cross_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")
scores = cross_model.predict([(query, i) for i in top_80_documents])
print(scores)

In [ ]:
# Pair each document with its cross-encoder score
ranked_results = list(zip(top_80_documents, scores))

# Sort by score - highest first
ranked_results = sorted(
    ranked_results,
    key=lambda x: x[1],
    reverse=True
)

# Get the top 10 chunks
top_chunks = [
    document
    for document, score in ranked_results[:30]
]

print(top_chunks)

In [ ]:
final_passing = top_chunks[0:11]

GENERATION SEGMENT


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load model
model_name = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

In [ ]:
# Combine your final retrieved chunks
context = "\n\n".join(final_passing)

In [ ]:

# RAG prompt
messages = [
    {
        "role": "system",
        "content": """You are a document question-answering system.

Answer the question using ONLY the provided context.

Return ONLY the final answer.

IMPORTANT:
- Give the COMPLETE answer.
- Do not omit percentages, monetary amounts, conditions, alternatives,
  exceptions, or qualifiers.
- If the answer contains an "or", "and", or "whichever is higher/lower"
  condition, include the entire condition.
- Preserve the exact meaning of the source.
- Do not provide reasoning or analysis."""
 },
    {
        "role": "user",
        "content": f"""/no_think

Context:
{context}

Question:
{query}

Final answer:"""
    }
]

In [ ]:
# Apply Qwen chat template
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

# Generate
outputs = model.generate(
    **inputs,
    max_new_tokens=700
)


In [ ]:
# Decode only generated answer
answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(answer)